In [1]:
import numpy as np
import pandas as pd
import math
import os
import glob
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output, display

# 显式声明模式
%matplotlib inline 

# ==========================================
# 1. 基础路径配置与数据持久化加载
# ==========================================
base_path = r'D:\code\data\green'
search_pattern = os.path.join(base_path, '202*.csv')
file_list = sorted(glob.glob(search_pattern))

if not file_list:
    print("❌ 未找到数据文件，请检查路径！")
else:
    print("⏳ 正在读取并拼接原始 CSV 数据，请稍候...")
    df_list = [pd.read_csv(f) for f in file_list]
    df = pd.concat(df_list, ignore_index=True)
    
    # 清理多余列并处理缺失值
    columns_to_drop = ['clean_green1', 'butter_green1']
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')
    df['green1'] = df['green1'].ffill().fillna(0)  

    # 反转信号极性
    INVERT_POLARITY = True
    polarity_multiplier = -1.0 if INVERT_POLARITY else 1.0
    raw_signal = df['green1'].values * polarity_multiplier
    
    total_points = len(raw_signal)
    fs = 100  # 采样率 100Hz
    print(f"✅ 数据加载就绪！总行数: {total_points}，总时长: {total_points/fs:.1f} 秒")

⏳ 正在读取并拼接原始 CSV 数据，请稍候...
✅ 数据加载就绪！总行数: 8262961，总时长: 82629.6 秒


In [10]:
import numpy as np
import math

def run_one_pass_filter(raw_y, fs=100, window_sec=10.0, q=201, rho=0, overlap=0.5):
    """
    终极流式处理管线：重叠分块 + 去趋势 + 汉宁窗交叉淡化
    """
    chunk_size = int(fs * window_sec)
    step_size = int(chunk_size * (1 - overlap))  # 默认 50% 重叠，步进 5 秒
    
    # 构建标准的汉宁窗 (Hann Window)，用于交叉淡化
    # 加一个小 epsilon 防止除零，虽然严格数学上不需要，但工程上更稳健
    window = np.hanning(chunk_size)
    
    # 预先构建标准窗口的 Phi 矩阵和 W 矩阵
    t_chunk = np.linspace(0, window_sec, chunk_size, endpoint=False)
    Phi = np.zeros((chunk_size, q))
    Phi[:, 0] = 1 / np.sqrt(window_sec)

    W = np.zeros((q, q))
    for k in range(1, (q + 1) // 2):
        freq = k / window_sec
        Phi[:, 2 * k - 1] = np.sqrt(2 / window_sec) * np.cos(2 * np.pi * freq * t_chunk)
        Phi[:, 2 * k] = np.sqrt(2 / window_sec) * np.sin(2 * np.pi * freq * t_chunk)
        penalty = (2 * np.pi * freq) ** 4
        W[2 * k - 1, 2 * k - 1] = penalty
        W[2 * k, 2 * k] = penalty

    H_hat = np.eye(q)
    # 计算核心的逆矩阵
    inv_mat = np.linalg.inv(H_hat + rho * W)

    # 初始化结果数组和用于归一化的权重数组 (Overlapped-Add 机制)
    ac_signal = np.zeros_like(raw_y, dtype=np.float64)
    weight_sum = np.zeros_like(raw_y, dtype=np.float64)

    # 计算总步数
    num_steps = math.ceil((len(raw_y) - chunk_size) / step_size) + 1
    # 确保至少走一步
    num_steps = max(1, num_steps)

    for i in range(num_steps):
        start_idx = i * step_size
        end_idx = start_idx + chunk_size
        
        # 处理最后一个可能不完整的块
        if end_idx > len(raw_y):
            end_idx = len(raw_y)
            
        current_len = end_idx - start_idx
        chunk_raw = raw_y[start_idx:end_idx]

        # ✅ 第一重保护：局部线性去趋势 (Detrend)，消除严重倾斜造成的两端飞点
        x_idx = np.arange(current_len)
        poly_coef = np.polyfit(x_idx, chunk_raw, 1) 
        trend = np.polyval(poly_coef, x_idx)        
        chunk_ac = chunk_raw - trend

        # 核心投影重构计算
        if current_len == chunk_size:
            G = Phi.T @ chunk_ac
            a_hat = inv_mat @ (G / chunk_size)
            reconstructed = Phi @ a_hat
            
            # ✅ 第二重保护：重叠相加 (Overlap-Add) 与 窗口加权 (Windowing)
            ac_signal[start_idx:end_idx] += reconstructed * window
            weight_sum[start_idx:end_idx] += window
            
        else:
            # 尾部处理（不完整窗口）
            t_temp = t_chunk[:current_len]
            Phi_temp = np.zeros((current_len, q))
            Phi_temp[:, 0] = 1 / np.sqrt(window_sec)
            for k in range(1, (q + 1) // 2):
                freq = k / window_sec
                Phi_temp[:, 2 * k - 1] = np.sqrt(2 / window_sec) * np.cos(2 * np.pi * freq * t_temp)
                Phi_temp[:, 2 * k] = np.sqrt(2 / window_sec) * np.sin(2 * np.pi * freq * t_temp)

            G_temp = Phi_temp.T @ chunk_ac
            a_hat_temp = inv_mat @ (G_temp / current_len)
            reconstructed = Phi_temp @ a_hat_temp
            
            # 尾部也应用相应的窗口片段
            window_temp = window[:current_len]
            ac_signal[start_idx:end_idx] += reconstructed * window_temp
            weight_sum[start_idx:end_idx] += window_temp

    # 最后，除以权重总和进行归一化。
    # 对于 50% 重叠的汉宁窗，中间大部分区域的 weight_sum 几乎正好等于 1.0
    # 这一步是为了防止边缘区域或重叠率改变时出现幅度失真
    mask = weight_sum > 1e-8
    ac_signal[mask] = ac_signal[mask] / weight_sum[mask]
    
    # 对于未能被窗口覆盖到的极少部分（通常是信号最末尾），直接保持为0或保持原始AC
    ac_signal[~mask] = 0.0 

    return ac_signal

In [11]:
# 初始化全局缓存变量，避免无意义的重复计算
_cached_q = None
_cached_rho = None
_cached_ac = None

def interactive_tuner_plot(q, rho, start_idx, window_size):
    global _cached_q, _cached_rho, _cached_ac
    
    # 避免前端 DOM 堆叠卡死
    clear_output(wait=True) 
    
    # 【智能缓存控制】只有当算法关键参数 q 或 rho 改变时，才重新运行矩阵滤波
    if _cached_ac is None or _cached_q != q or _cached_rho != rho:
        print(f"⚙️ 监测到参数变更 (q={q}, rho={rho})，正在重新计算流式信号重构...")
        _cached_ac = run_one_pass_filter(raw_signal, fs=100, window_sec=10.0, q=q, rho=rho)
        _cached_q = q
        _cached_rho = rho
    
    # 切片当前显示区域
    end_idx = min(start_idx + window_size, total_points)
    time_axis = np.arange(start_idx, end_idx) / fs
    
    raw_slice = raw_signal[start_idx:end_idx]
    ac_slice = _cached_ac[start_idx:end_idx]
    
    # 显式声明图表并设置内存回收
    plt.close('all') 
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
    
    # -------- 子图 1: 原始 PPG --------
    ax1.plot(time_axis, raw_slice, color='gray', linewidth=1.5, label='Raw PPG Signal')
    ax1.set_title(f'One-pass Nonparametric Regression Tuner (Time: {start_idx/fs:.1f}s to {end_idx/fs:.1f}s)', fontsize=14, pad=10)
    ax1.set_ylabel('Raw Amplitude', fontsize=12)
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # -------- 子图 2: 滤波后的交流波形 --------
    ax2.plot(time_axis, ac_slice, color='red', linewidth=2.5, label=f'Reconstructed AC (q={q}, rho={rho})')
    ax2.set_xlabel('Time (Seconds)', fontsize=12)
    ax2.set_ylabel('AC Amplitude', fontsize=12)
    ax2.axhline(0, color='black', linestyle='--', linewidth=0.8) 
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    plt.close(fig) 

# ================= 交互控件构建 =================
max_start = max(0, total_points - 500) 

# 【修改点】算法调参控件改为精确选项的下拉菜单
dropdown_q = widgets.Dropdown(
    options=[101, 201, 301, 401, 501], 
    value=201, 
    description='基底数量 q:'
)

dropdown_rho = widgets.Dropdown(
    options=[
        ('0 (无惩罚)', 0), 
        ('1e-6 (轻微)', 1e-6), 
        ('2e-6', 2e-6), 
        ('3e-6', 3e-6), 
        ('4e-6', 4e-6),
        ('4.5e-6', 4.5e-6),
        ('5e-6', 5e-6), 
        ('7e-6', 7e-6), 
        ('1e-5 (较平滑)', 1e-5)
    ], 
    value=3e-6, # 默认停在中间位置，方便对比
    description='惩罚系数 rho:'
)

# 视口控制控件保持不变
slider_start = widgets.IntSlider(min=0, max=max_start, step=200, value=0, description='起点索引:', continuous_update=False, layout=widgets.Layout(width='400px'))
slider_window = widgets.IntSlider(min=200, max=5000, step=100, value=1000, description='窗口大小:', continuous_update=False, layout=widgets.Layout(width='400px'))

# 将所有控件与函数绑定
out = widgets.interactive_output(
    interactive_tuner_plot, 
    {'q': dropdown_q, 'rho': dropdown_rho, 'start_idx': slider_start, 'window_size': slider_window}
)

# 界面分行美化排版
ui_algo = widgets.HBox([dropdown_q, dropdown_rho])
ui_view = widgets.HBox([slider_start, slider_window])
ui_panel = widgets.VBox([ui_algo, ui_view])

# 渲染界面
display(ui_panel, out)

Output(outputs=({'name': 'stdout', 'text': '⚙️ 监测到参数变更 (q=201, rho=3e-06)，正在重新计算流式信号重构...\n', 'output_type': '…

In [13]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from datetime import datetime, timedelta
import os

def extract_ppg_features(ac_signal, fs=100, start_time_str="06172024 080000"):
    """
    流式/全局 PPG 信号特征提取引擎
    
    参数:
    ac_signal: 经过 One-pass 滤波后的高质量交流信号
    fs: 采样率 (默认 100 Hz)
    start_time_str: 数据记录的起始时间，格式为 "月日年 时分秒" (MMDDYYYY HHMMSS)
    """
    print("🚀 开始提取心跳波形特征...")
    
    # 解析起始时间
    try:
        dt_start = datetime.strptime(start_time_str, "%m%d%Y %H%M%S")
    except ValueError:
        print("⚠️ 时间格式错误，将使用当前系统时间作为基准。请确保格式为 '06172024 143000'")
        dt_start = datetime.now()

    # ==========================================
    # 1. 拓扑寻峰与寻谷 (Peak & Valley Detection)
    # ==========================================
    # 正常心率范围 40 ~ 200 BPM。
    # 200 BPM 对应间隔 0.3 秒 (fs=100下为 30 个数据点)。所以我们设定 distance=30 防误触
    # prominence(突起度) 利用信号的标准差动态自适应，过滤掉微小的高频涟漪
    distance_min = int(fs * 0.3) 
    dynamic_prominence = np.std(ac_signal) * 0.2
    
    # 找波峰 (Systolic Peaks)
    peaks, _ = find_peaks(ac_signal, distance=distance_min, prominence=dynamic_prominence)
    
    # 找波谷 (Diastolic Valleys / Foot) - 通过将信号翻转来寻找
    valleys, _ = find_peaks(-ac_signal, distance=distance_min, prominence=dynamic_prominence)

    # ==========================================
    # 2. 逐拍特征提取 (Beat-by-beat Extraction)
    # ==========================================
    features_list = []
    
    # 舍弃第一个和最后一个波峰，因为它们可能没有完整的波谷包围，或者无法计算 RRI
    for i in range(1, len(peaks) - 1):
        peak_idx = peaks[i]
        prev_peak_idx = peaks[i-1]
        
        # 寻找当前波峰左侧最近的波谷 (心跳起点/上升支起点)
        left_valleys = valleys[valleys < peak_idx]
        if len(left_valleys) == 0: continue
        onset_idx = left_valleys[-1]
        
        # 寻找当前波峰右侧最近的波谷 (心跳终点/下降支终点)
        right_valleys = valleys[valleys > peak_idx]
        if len(right_valleys) == 0: continue
        end_idx = right_valleys[0]
        
        # === A. 时间与时间戳特征 ===
        # 计算该波峰对应的绝对时间 (月日年 时分秒)
        peak_time_exact = dt_start + timedelta(seconds=int(peak_idx) / fs)
        timestamp_str = peak_time_exact.strftime("%m%d%Y %H%M%S")
        
        # RRI (Peak-to-Peak Interval) 峰峰期，单位：毫秒 (ms)
        rri_ms = (peak_idx - prev_peak_idx) * (1000.0 / fs)
        
        # 瞬时心率 (Instantaneous Heart Rate)
        hr_bpm = 60000.0 / rri_ms if rri_ms > 0 else 0
        
        # 收缩期时间 (起点到波峰)，单位：毫秒
        systolic_time_ms = (peak_idx - onset_idx) * (1000.0 / fs)
        
        # 舒张期时间 (波峰到终点)，单位：毫秒
        diastolic_time_ms = (end_idx - peak_idx) * (1000.0 / fs)
        
        # === B. 幅度特征 ===
        peak_val = ac_signal[peak_idx]
        onset_val = ac_signal[onset_idx]
        end_val = ac_signal[end_idx]
        
        # 脉搏波振幅 (Pulse Amplitude)
        pulse_amplitude = peak_val - onset_val
        
        # === C. 形态学特征 (前几点，后几点) ===
        # 提取波峰前后固定的位置，用于机器学习刻画波峰锐度。如果越界则用 NaN 填充
        morph = {}
        offsets = [-10, -5, -3, 3, 5, 10]  # 分别提取前10,前5,前3，后3,后5,后10个采样点
        for offset in offsets:
            target_idx = peak_idx + offset
            col_name = f"morph_{'pre' if offset < 0 else 'post'}_{abs(offset)}"
            if 0 <= target_idx < len(ac_signal):
                # 记录相对振幅 (减去基底起点，更能反映真实形态)
                morph[col_name] = ac_signal[target_idx] - onset_val
            else:
                morph[col_name] = np.nan
        
        # === 汇总当前心跳的所有特征 ===
        beat_features = {
            "timestamp": timestamp_str,         # 时间戳 (月日年 时分秒)
            "peak_idx": peak_idx,               # 波峰索引
            "onset_idx": onset_idx,             # 起点波谷索引
            "end_idx": end_idx,                 # 终点波谷索引
            "rri_ms": round(rri_ms, 2),         # RR间期 (毫秒)
            "hr_bpm": round(hr_bpm, 1),         # 瞬时心率 (BPM)
            "sys_time_ms": round(systolic_time_ms, 2), # 收缩时间 (毫秒)
            "dia_time_ms": round(diastolic_time_ms, 2),# 舒张时间 (毫秒)
            "pulse_amp": round(pulse_amplitude, 4),    # 搏动绝对振幅
            **morph                             # 展开形态学字典
        }
        features_list.append(beat_features)

    # 转化为强大的 Pandas DataFrame 进行输出
    df_features = pd.DataFrame(features_list)
    print(f"✅ 特征提取完成！成功提取 {len(df_features)} 个完整心跳周期。")
    return df_features

# ==========================================
# 3. 实际调用演示 (如何与前面的滤波管线结合)
# ==========================================
if __name__ == "__main__":
    # 请根据你的 CSV 数据实际开始时间修改这里的字符串
    REAL_START_TIME = "05202024 143000" 
    signal_to_process = None

    # 1. 尝试从 Jupyter 内存加载 (如果是依次运行的 Cell)
    if 'ac_signal' in locals() or 'ac_signal' in globals():
        signal_to_process = ac_signal
        print("✅ 检测到内存中的 ac_signal，准备提取特征...")
    else:
        print("⚠️ 内存中未检测到 ac_signal 变量。")
        
        # 2. 尝试从磁盘加载之前保存的 .npy 文件
        # 注意：这里的路径与之前数据加载的 base_path 保持一致
        npy_path = r'D:\code\data\green\ac_signal_onepass.npy'
        
        if os.path.exists(npy_path):
            signal_to_process = np.load(npy_path)
            print(f"✅ 成功从磁盘读取数据文件: {npy_path}")
        else:
            # 3. 如果文件也没有，生成一段模拟信号用以演示程序逻辑
            print("⚠️ 未找到本地数据文件。生成一段包含基线漂移的模拟 PPG 信号进行演示...")
            fs_mock = 100
            t = np.linspace(0, 10, fs_mock * 10)  # 10秒时长
            # 基础心跳 (1.2Hz，约 72 BPM) + 随机噪声
            signal_to_process = (np.sin(2 * np.pi * 1.2 * t) * 500) + np.random.normal(0, 20, len(t))
    
    # 执行特征提取
    df_heartbeats = extract_ppg_features(signal_to_process, fs=100, start_time_str=REAL_START_TIME)
    
    # 兼容 Jupyter 的 display 和纯 Python 的 print
    try:
        display(df_heartbeats.head())
    except NameError:
        print(df_heartbeats.head().to_string())
    
    # 将提取到的极高质量心跳特征保存为 CSV，供后续算法（如随机森林、神经网络分析、HRV分析）使用
    # output_csv = r'D:\code\data\green\heartbeat_features_q101_rho7e-6.csv'
    # df_heartbeats.to_csv(output_csv, index=False)
    # print(f"💾 特征已保存至: {output_csv}")

⚠️ 内存中未检测到 ac_signal 变量。
✅ 成功从磁盘读取数据文件: D:\code\data\green\ac_signal_onepass.npy
🚀 开始提取心跳波形特征...
✅ 特征提取完成！成功提取 34785 个完整心跳周期。


,timestamp,peak_idx,onset_idx,end_idx,rri_ms,hr_bpm,sys_time_ms,dia_time_ms,pulse_amp,morph_pre_10,morph_pre_5,morph_pre_3,morph_post_3,morph_post_5,morph_post_10
0,05202024 143000,46,4,71,370.0,162.2,420.0,250.0,2531.0687,2460.008647,1086.189042,1326.746714,1586.580559,672.052466,2577.358464
1,05202024 143000,81,71,141,350.0,171.4,100.0,600.0,27496.8446,0.000000,13677.362958,22412.525117,24580.208364,22604.187134,24114.344294
2,05202024 143001,125,71,141,440.0,136.4,540.0,160.0,40882.0738,25409.697864,25385.237438,31622.317825,34062.145083,20822.826494,-8619.462395
3,05202024 143001,170,141,184,450.0,133.3,290.0,140.0,54139.6687,40522.638228,42920.535147,50149.227655,49979.703039,45212.181286,10787.361252
4,05202024 143002,239,214,264,690.0,87.0,250.0,250.0,71524.9685,57019.115059,67232.396827,69889.884921,69795.675560,69101.303189,71182.404629
